In [15]:
import ollama
import json
import re

# -----------------------------
# 1. CALL LLM
# -----------------------------
def call_llm(user_input):
    prompt = f"""
You are a system dynamics expert. Your job is to extract stock-and-flow structures from text descriptions.

Text:
\"\"\"{user_input}\"\"\"

REQUIRED OUTPUT FORMAT - You MUST return valid JSON with ALL of these keys:
{{
  "stocks": ["Name of accumulating quantity"],
  "inflows": ["Name of rate that increases stock"],
  "outflows": ["Name of rate that decreases stock"],
  "auxiliary": ["Name of helper variables"],
  "equations": {{
    "stock": "INTEG(inflow_name - outflow_name, initial_value)",
    "inflow": "mathematical formula",
    "outflow": "mathematical formula",
    "auxiliary": "formula"
  }}
}}
 
RULES:
1. STOCKS accumulate over time (e.g., "Views", "Water Level", "Population")
2. INFLOWS are rates that ADD to stocks (e.g., "New Views", "Filling Rate")
3. OUTFLOWS are rates that SUBTRACT from stocks (e.g., "View Decay", "Leakage")
4. AUXILIARY are helper variables that aren't stocks or flows
5. Each stock must have an INTEG() equation
6. Return ONLY the JSON object, no markdown, no code blocks, no explanations
 
EXAMPLE:
Input: "When a post gets more views, people share it more, bringing in even more views."
Output:
{{
  "stocks": ["Post Views"],
  "inflows": ["New Views from Shares"],
  "outflows": ["View Decay"],
  "auxiliary": ["Share Probability", "Decay Rate"],
  "equations": {{
    "Post Views": "INTEG(New Views from Shares - View Decay, 0)",
    "New Views from Shares": "Post Views * Share Probability",
    "View Decay": "Post Views * Decay Rate",
    "Share Probability": "0.1",
    "Decay Rate": "0.05"
  }}
}}

Now analyze the given text and return JSON only.
"""
    try:
        response = ollama.chat(
            model='qwen2.5-coder:7b',
            messages=[{"role": "user", "content": prompt}]
        )
        return response['message']['content']

    except Exception as e:
        print("LLM call failed:", e)
        return ""

# -----------------------------
# 2. PARSE JSON SAFELY
# -----------------------------
def parse_json(response_text):
    try:
        # Try direct parsing first
        return json.loads(response_text)
    except json.JSONDecodeError:
        pass
    
    try:
        # Extract JSON from messy output (handles markdown code blocks)
        json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group(1))
    except json.JSONDecodeError:
        pass
    
    try:
        # Extract raw JSON object
        json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except json.JSONDecodeError as e:
        print("JSON parsing error:", e)
    
    return None

# -----------------------------
# 2B. EXTRACT MISSING FIELDS FROM JSON
# -----------------------------
def extract_missing_fields(data):
    """If stocks/inflows/outflows are missing, try to extract from equations"""
    if not data or not data.get("equations"):
        return data
    
    equations = data["equations"]
    
    # Extract stocks (those with INTEG)
    if not data.get("stocks"):
        stocks = [var for var, eq in equations.items() if "INTEG" in eq.upper()]
        if stocks:
            data["stocks"] = stocks
    
    # Extract inflows (mentioned in INTEG right side before minus)
    if not data.get("inflows"):
        inflows = set()
        for var, eq in equations.items():
            if "INTEG" in eq.upper():
                # Parse "INTEG(inflow - outflow, init)"
                match = re.search(r'INTEG\s*\(\s*([^-]+?)\s*-', eq, re.IGNORECASE)
                if match:
                    inflows.add(match.group(1).strip())
        if inflows:
            data["inflows"] = list(inflows)
    
    # Extract outflows (mentioned in INTEG right side after minus)
    if not data.get("outflows"):
        outflows = set()
        for var, eq in equations.items():
            if "INTEG" in eq.upper():
                # Parse "INTEG(inflow - outflow, init)"
                match = re.search(r'INTEG\s*\([^-]*-\s*([^,]+?)\s*,', eq, re.IGNORECASE)
                if match:
                    outflows.add(match.group(1).strip())
        if outflows:
            data["outflows"] = list(outflows)
    
    return data
 

# -----------------------------
# 3. VALIDATION
# -----------------------------
def validate(data):
    errors = []
 
    if not data:
        errors.append("No data parsed")
        return errors
 
    # Check for required keys (using plural forms to match prompt)
    if not data.get("stocks") or len(data.get("stocks", [])) == 0:
        errors.append("Missing stocks")
 
    if not data.get("inflows") or len(data.get("inflows", [])) == 0:
        errors.append("No inflows defined")
 
    if not data.get("outflows") or len(data.get("outflows", [])) == 0:
        errors.append("No outflows defined")
 
    if not data.get("equations") or len(data.get("equations", {})) == 0:
        errors.append("No equations provided")

    return errors


# -----------------------------
# 4. GENERATE MDL
# -----------------------------
def generate_mdl(data):
    lines = []

    # Header
    lines.append("{UTF-8}")

    equations = data.get("equations", {})

    def format_var(name):
        return name.lower()

    # Generate variables
    for var, eq in equations.items():
        var_name = format_var(var)
        eq_formatted = eq.replace("*", "*").replace("  ", " ")

        lines.append(f"{var_name} = {eq_formatted}")
        lines.append("~")
        lines.append("~ |")
        lines.append("")

    # Control section
    lines.append("********************************************************")
    lines.append(".Control")
    lines.append("********************************************************~")
    lines.append("Simulation Control Parameters")
    lines.append("|")
    lines.append("")
    lines.append("FINAL TIME  = 100")
    lines.append("~ Month")
    lines.append("~ The final time for the simulation.")
    lines.append("|")
    lines.append("")
    lines.append("INITIAL TIME  = 0")
    lines.append("~ Month")
    lines.append("~ The initial time for the simulation.")
    lines.append("|")
    lines.append("")
    lines.append("SAVEPER  =")
    lines.append("        TIME STEP")
    lines.append("~ Month [0,?]")
    lines.append("~ The frequency with which output is stored.")
    lines.append("|")
    lines.append("")
    lines.append("TIME STEP  = 1")
    lines.append("~ Month [0,?]")
    lines.append("~ The time step for the simulation.")
    lines.append("|")

    return "\n".join(lines)


# -----------------------------
# 5. FULL PIPELINE
# -----------------------------
def run_pipeline(user_input):
    print("🔹 Input:", user_input)
    print("\n" + "="*70)
 
    raw = call_llm(user_input)
    print("\n🔹 Raw LLM Output:")
    print(raw)
    print("="*70)
 
    parsed = parse_json(raw)
    print("\n🔹 Parsed JSON (before extraction):")
    if parsed:
        print(json.dumps(parsed, indent=2))
    else:
        print("Failed to parse JSON")
    
    # Try to extract missing stocks/inflows/outflows from equations
    parsed = extract_missing_fields(parsed)
    print("\n🔹 After field extraction:")
    if parsed:
        print(json.dumps(parsed, indent=2))
    print("="*70)
 
    errors = validate(parsed)
 
    if errors:
        print("\n❌ Validation Errors:")
        for error in errors:
            print(f"  - {error}")
        return None
 
    mdl = generate_mdl(parsed)
    print("\n✅ MDL Output:")
    print(mdl)
    print("="*70)
 
    return parsed, mdl

In [16]:
run_pipeline("More births increase population, more deaths decrease population")

🔹 Input: More births increase population, more deaths decrease population


🔹 Raw LLM Output:
{
  "stocks": ["Population"],
  "inflows": ["Births"],
  "outflows": ["Deaths"],
  "auxiliary": [],
  "equations": {
    "Population": "INTEG(Births - Deaths, 0)",
    "Births": "",
    "Deaths": ""
  }
}

🔹 Parsed JSON (before extraction):
{
  "stocks": [
    "Population"
  ],
  "inflows": [
    "Births"
  ],
  "outflows": [
    "Deaths"
  ],
  "auxiliary": [],
  "equations": {
    "Population": "INTEG(Births - Deaths, 0)",
    "Births": "",
    "Deaths": ""
  }
}

🔹 After field extraction:
{
  "stocks": [
    "Population"
  ],
  "inflows": [
    "Births"
  ],
  "outflows": [
    "Deaths"
  ],
  "auxiliary": [],
  "equations": {
    "Population": "INTEG(Births - Deaths, 0)",
    "Births": "",
    "Deaths": ""
  }
}

✅ MDL Output:
{UTF-8}
population = INTEG(Births - Deaths, 0)
~
~ |

births = 
~
~ |

deaths = 
~
~ |

********************************************************
.Control
***********

({'stocks': ['Population'],
  'inflows': ['Births'],
  'outflows': ['Deaths'],
  'auxiliary': [],
  'equations': {'Population': 'INTEG(Births - Deaths, 0)',
   'Births': '',
   'Deaths': ''}},
 '{UTF-8}\npopulation = INTEG(Births - Deaths, 0)\n~\n~ |\n\nbirths = \n~\n~ |\n\ndeaths = \n~\n~ |\n\n********************************************************\n.Control\n********************************************************~\nSimulation Control Parameters\n|\n\nFINAL TIME  = 100\n~ Month\n~ The final time for the simulation.\n|\n\nINITIAL TIME  = 0\n~ Month\n~ The initial time for the simulation.\n|\n\nSAVEPER  =\n        TIME STEP\n~ Month [0,?]\n~ The frequency with which output is stored.\n|\n\nTIME STEP  = 1\n~ Month [0,?]\n~ The time step for the simulation.\n|')